In [0]:
#Write a pyspark code to find the number of employees in a school

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, max, sum, when
from pyspark.sql.window import Window
import datetime

# Initialize SparkSession
spark = SparkSession.builder.appName("SchoolEmployees").getOrCreate()

# Sample data for school employee actions
data = [
    (1, 'in', datetime.datetime(2023, 10, 1, 8, 0, 0)),
    (1, 'out', datetime.datetime(2023, 10, 1, 12, 0, 0)),
    (2, 'in', datetime.datetime(2023, 10, 1, 9, 0, 0)),
    (3, 'in', datetime.datetime(2023, 10, 1, 8, 30, 0)),
    (3, 'out', datetime.datetime(2023, 10, 1, 10, 30, 0)),
    (3, 'in', datetime.datetime(2023, 10, 1, 11, 0, 0)),
    (4, 'out', datetime.datetime(2023, 10, 1, 9, 30, 0)),
    (4, 'in', datetime.datetime(2023, 10, 1, 10, 0, 0)),
]

# Define Schema
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType
schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("action", StringType(), True),
    StructField("time", TimestampType(), True)
]) 

# Create DataFrame
df = spark.createDataFrame(data, schema)

# Define a window specification to sort actions by time for each employee
window_spec = Window.partitionBy("emp_id").orderBy("time")

# Assign a cumulative 'in' and 'out' count for each employee
df_with_counts = df.withColumn(
    "net_in", 
    sum(when(col("action") == "in", 1).otherwise(-1)).over(window_spec)
)

# Select the latest record for each employee to determine their current status
latest_status = df_with_counts.withColumn(
    "latest_time", max("time").over(Window.partitionBy("emp_id"))
).filter(col("time") == col("latest_time"))

# Filter employees who are currently "in"
current_employees = latest_status.filter(col("net_in") > 0)

# Count the number of employees inside the school
num_employees_inside = current_employees.count()

print(f"Number of employees currently in the school: {num_employees_inside}")


Number of employees currently in the school: 2
